# Notebook 09 — CNN Baseline for Image Classification

## Learning Objectives
- Understand how a simple two-block CNN classifies images.
- Load Fashion-MNIST with `src.data.vision_datasets.load_fashion_mnist`.
- Train `SimpleCNN` for 3 epochs and track accuracy per epoch.
- Visualise a sample image grid and a confusion matrix.
- Establish a CNN performance baseline before comparing with Vision Transformer in Notebook 10.
- Inspect failure cases and understand when CNNs struggle.

## Big Picture
Before the Vision Transformer era, CNNs dominated image recognition. Their inductive biases
(locality, translation equivariance) made them data-efficient for structured images. This notebook
trains a compact `SimpleCNN` on Fashion-MNIST and saves all metrics so that Notebook 10 can do a
fair apples-to-apples comparison against ViT.

## Dataset
**Fashion-MNIST** — 70,000 greyscale 28×28 images in 10 clothing categories.  
Split: 54,000 train / 6,000 val / 10,000 test (fixed split from torchvision).

## Architecture
```
Input [B, 1, 28, 28]
  └─ Conv2d(1→32, 3×3, pad=1) + BN + ReLU + MaxPool(2×2)  → [B, 32, 14, 14]
  └─ Conv2d(32→64, 3×3, pad=1) + BN + ReLU + MaxPool(2×2) → [B, 64, 7, 7]
  └─ Flatten → [B, 3136]
  └─ Linear(3136→256) + ReLU + Dropout(0.3)
  └─ Linear(256→10)   → logits [B, 10]
```

## Theory
**Convolutional layer**: applies a learnable filter that slides over the image, detecting local
patterns (edges, textures) regardless of position — *translation equivariance*.

**MaxPooling**: halves spatial size, makes features robust to small shifts.

**BatchNorm**: normalises activations within a mini-batch, stabilising training and allowing
higher learning rates.

**Cross-entropy loss**:
$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{p}_c)$$

where $y_c$ is the one-hot target and $\hat{p}_c$ is the softmax probability.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary
from src.utils.paths import RUNS_VIT_CLS_DIR
from src.data.vision_datasets import load_fashion_mnist, FASHION_MNIST_CLASSES
from src.models.cnn_baseline import SimpleCNN
from src.visualization.plots import show_image_grid, plot_training_curves, plot_confusion_matrix

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_VIT_CLS_DIR, clean=False)  # shared dir; don't clean ViT outputs
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')
print(f'Classes : {FASHION_MNIST_CLASSES}')

## Dataset
### 2. Load Fashion-MNIST

In [ ]:
BATCH_SIZE = 64
NUM_EPOCHS = 3
LR = 1e-3
IMAGE_SIZE = 28
NUM_CLASSES = 10

train_loader, val_loader, test_loader = load_fashion_mnist(
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

# Inspect one batch
images, labels = next(iter(train_loader))
print(f'Image batch shape : {images.shape}  (B, C, H, W)')
print(f'Label batch shape : {labels.shape}')
print(f'Pixel range       : [{images.min():.3f}, {images.max():.3f}]  (normalised)')
print(f'Train batches : {len(train_loader)}  |  Val : {len(val_loader)}  |  Test : {len(test_loader)}')

In [ ]:
# Visualise a grid of 16 sample images
imgs_np = images[:16].numpy()  # [16, 1, 28, 28]
# Unnormalise for display: Fashion-MNIST mean=0.286, std=0.353
imgs_display = imgs_np * 0.353 + 0.286
imgs_display = np.clip(imgs_display, 0, 1)
label_names = [FASHION_MNIST_CLASSES[l.item()] for l in labels[:16]]

show_image_grid(
    imgs_display,
    labels=label_names,
    save_path=run_dir / 'cnn_sample_grid.png',
    ncols=8,
    title='Fashion-MNIST Sample Images',
    cmap='gray',
)
print('Sample grid saved.')

### 3. Build the SimpleCNN Model

In [ ]:
model = SimpleCNN(
    in_channels=1,
    num_classes=NUM_CLASSES,
    image_size=IMAGE_SIZE,
    dropout=0.3,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters: {n_params:,}')

# Shape trace through the network
x_sample = images[:4].to(device)   # [4, 1, 28, 28]
with torch.no_grad():
    logits = model(x_sample)        # [4, 10]

print(f'Input  shape : {x_sample.shape}   (batch, channels, H, W)')
print(f'Output shape : {logits.shape}   (batch, num_classes)')
print(f'Logit range  : [{logits.min():.3f}, {logits.max():.3f}]')

## Sanity Checks

In [ ]:
# 1. Output shape check
assert logits.shape == (4, NUM_CLASSES), f'Unexpected logits shape: {logits.shape}'
print('Output shape check passed.')

# 2. No NaN
assert not torch.isnan(logits).any(), 'NaN in logits!'
print('No NaN in output.')

# 3. Softmax sums to 1
probs = F.softmax(logits, dim=-1)
assert torch.allclose(probs.sum(dim=-1), torch.ones(4, device=device), atol=1e-5)
print('Softmax sum-to-1 check passed.')

# 4. Random-init accuracy ~ 1/10 = 10%
preds = logits.argmax(dim=-1)
acc = (preds == labels[:4].to(device)).float().mean().item()
print(f'Random-init accuracy on 4 samples: {acc:.2f}  (expected ~0.10 on average)')

print('All sanity checks passed!')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training phase ──────────────────────────────────────────────────────
    model.train()
    total_loss, n_batches = 0.0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)                         # [B, 10]
        loss = F.cross_entropy(out, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    train_losses.append(total_loss / n_batches)

    # ── Validation phase ─────────────────────────────────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            v_loss += F.cross_entropy(out, lbls).item()
            v_correct += (out.argmax(-1) == lbls).sum().item()
            v_total += lbls.size(0)
    val_losses.append(v_loss / len(val_loader))
    val_accs.append(v_correct / v_total)

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train_loss={train_losses[-1]:.4f}  '
          f'val_loss={val_losses[-1]:.4f}  '
          f'val_acc={val_accs[-1]:.4f}')

print('Training complete!')

## Evaluation

In [ ]:
# Evaluate on test set
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        all_preds.append(out.argmax(-1).cpu())
        all_labels.append(lbls)

all_preds  = torch.cat(all_preds).numpy()   # [10000]
all_labels = torch.cat(all_labels).numpy()  # [10000]

test_acc = (all_preds == all_labels).mean()
print(f'Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

print('\nPer-class report:')
print(classification_report(all_labels, all_preds, target_names=FASHION_MNIST_CLASSES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plot_confusion_matrix(
    cm,
    class_names=FASHION_MNIST_CLASSES,
    save_path=run_dir / 'cnn_confusion_matrix.png',
    title='SimpleCNN — Confusion Matrix (Fashion-MNIST Test Set)',
)
print('Confusion matrix saved.')

# Save metrics JSON
metrics = {
    'test_accuracy': float(test_acc),
    'final_val_accuracy': float(val_accs[-1]),
    'final_train_loss': float(train_losses[-1]),
    'final_val_loss': float(val_losses[-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
}
save_metrics_json(metrics, run_dir / 'cnn_metrics.json')
print(f'Metrics saved to: {run_dir / "cnn_metrics.json"}')

## Visualization

In [ ]:
# Training curves
history = {
    'train_loss': train_losses,
    'val_loss': val_losses,
    'val_accuracy': val_accs,
}
plot_training_curves(
    history,
    save_path=run_dir / 'cnn_training_curve.png',
    title='SimpleCNN Training Curves (Fashion-MNIST)',
)
print('Training curve saved.')

In [ ]:
# Show sample predictions (correct and incorrect)
model.eval()
imgs_vis, lbls_vis = next(iter(test_loader))
with torch.no_grad():
    preds_vis = model(imgs_vis.to(device)).argmax(-1).cpu()

# Pick 8 images: 4 correct, 4 wrong
correct_idx = (preds_vis == lbls_vis).nonzero(as_tuple=True)[0][:4]
wrong_idx   = (preds_vis != lbls_vis).nonzero(as_tuple=True)[0][:4]
show_idx = torch.cat([correct_idx, wrong_idx])

imgs_show = (imgs_vis[show_idx].numpy() * 0.353 + 0.286).clip(0, 1)
titles = []
for i in show_idx:
    gt  = FASHION_MNIST_CLASSES[lbls_vis[i].item()]
    pr  = FASHION_MNIST_CLASSES[preds_vis[i].item()]
    ok  = 'OK' if lbls_vis[i] == preds_vis[i] else 'ERR'
    titles.append(f'{ok}: {pr}')

show_image_grid(
    imgs_show, labels=titles,
    save_path=run_dir / 'cnn_predictions.png',
    ncols=8,
    title='CNN Predictions (first 4 correct, last 4 wrong)',
    cmap='gray',
)
print('Prediction grid saved.')

## Failure Cases

**Common CNN failure patterns on Fashion-MNIST:**

1. **Shirt vs T-shirt/Pullover**: Very similar textures and shapes. CNNs rely on local texture patterns and miss
   the global structural differences (collar shape, seam placement).

2. **Sandal vs Sneaker vs Ankle boot**: All are footwear. The distinguishing features (ankle strap, sole height)
   are small and can be lost after two MaxPool operations that halve spatial resolution.

3. **Coat vs Dress**: Both have long silhouettes. Without attention to global shape, local CNN filters
   can confuse them, especially in unusual poses or when the image is noisy.

4. **Sensitivity to augmentation**: This baseline has no data augmentation. Random horizontal flip or
   slight rotation would improve robustness and reduce overfitting.

## Exercises

1. **Add data augmentation**: Apply `T.RandomHorizontalFlip()` and `T.RandomAffine(degrees=10)` in the
   training transform. Does test accuracy improve?

2. **Deeper network**: Add a third Conv block (64→128 filters) before the classifier. Count the new
   parameter count. Does accuracy improve? Does training time increase?

3. **Learning rate schedule**: Replace constant lr=1e-3 with `CosineAnnealingLR`. Plot the loss curve
   comparison. At what epoch does the cosine schedule help most?

4. **Per-class accuracy analysis**: From the confusion matrix, identify the two most-confused class pairs.
   Plot examples from each pair side-by-side. Can you tell them apart by eye?

5. **Feature map visualisation**: Extract the output of `conv_block1` for a single image and display all
   32 feature maps as a grid. Which filters detect edges? Which detect textures?

## Key Takeaways

- **CNNs exploit spatial locality**: filters slide over the image and detect local patterns independent
  of their position — a strong inductive bias that works well for natural images.
- **MaxPool + deeper channels** progressively build up from edges → textures → parts → objects.
- **BatchNorm** stabilises training and acts as mild regularisation.
- **Dropout** before the final linear layer prevents the classifier from co-adapting with individual neurons.
- With only 3 epochs, the CNN already reaches >85% accuracy on Fashion-MNIST — a testament to the
  efficiency of the CNN inductive bias.
- This baseline will be compared against ViT in Notebook 10.

## Saved Outputs

In [ ]:
# Save the session report
save_markdown_report(
    title='CNN Baseline — Fashion-MNIST Classification',
    summary=(
        f'SimpleCNN trained for {NUM_EPOCHS} epochs on Fashion-MNIST. '
        f'Test accuracy: {test_acc*100:.2f}%. '
        f'Model has {n_params:,} trainable parameters.'
    ),
    metrics=metrics,
    figures=['cnn_training_curve.png', 'cnn_confusion_matrix.png', 'cnn_predictions.png'],
    tables=[],
    output_path=run_dir / 'cnn_report.md',
    device=str(device),
    hyperparams={'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'lr': LR},
    architecture='Conv(1→32)+BN+ReLU+Pool → Conv(32→64)+BN+ReLU+Pool → Flatten → Linear(3136→256) → Linear(256→10)',
    loss_fn='CrossEntropyLoss',
)

update_runs_summary(
    session_name='cnn_baseline',
    report_path=run_dir / 'cnn_report.md',
    metrics=metrics,
    figures=['cnn_training_curve.png', 'cnn_confusion_matrix.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/cnn_metrics.json')
print(f'  {run_dir}/cnn_training_curve.png')
print(f'  {run_dir}/cnn_confusion_matrix.png')
print(f'  {run_dir}/cnn_report.md')